In [ ]:
# getting annotation in json format

import requests
import json

# ============================================================
# VEP: QUERY ALL 39 RSIDS AT ONCE
# ============================================================

rsids = df["RSID"].dropna().unique().tolist()

vep_url = "https://rest.ensembl.org/vep/human/id"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

payload = {
    "ids": rsids
}

print(f"Querying VEP for {len(rsids)} variants...")

r = requests.post(
    vep_url,
    headers=headers,
    json=payload,
    timeout=120
)

print("HTTP status:", r.status_code)

if r.status_code != 200:
    print("VEP error:")
    print(r.text)
    raise SystemExit

vep_data = r.json()

print(f"VEP returned {len(vep_data)} variants")

# Save raw output
with open("top39_VEP_raw.json", "w") as f:
    json.dump(vep_data, f, indent=2)

# Make dictionary by rsID
vep_results = {}

for result in vep_data:

    input_id = result.get("input")

    if input_id:
        vep_results[input_id] = result

print(
    f"Successfully annotated "
    f"{len(vep_results)} / {len(rsids)} variants"
)

# Show variants that weren't returned
missing = [
    rsid for rsid in rsids
    if rsid not in vep_results
]

if missing:
    print("\nMissing variants:")
    for rsid in missing:
        print(rsid)
```

Making it into a table:
```
import json
import pandas as pd

# ============================================================
# INPUT / OUTPUT
# ============================================================

INPUT = "top39_VEP_raw.json"
OUTPUT = "top39_VEP_annotation.tsv"

# ============================================================
# LOAD VEP JSON
# ============================================================

with open(INPUT, "r") as f:
    vep = json.load(f)

print(f"Loaded {len(vep)} VEP records")


# ============================================================
# HELPER FUNCTION
# ============================================================

def unique_join(values):
    """Join unique non-empty values with semicolons."""

    values = [
        str(x) for x in values
        if x is not None and str(x) not in ["", "nan", "None"]
    ]

    return ";".join(sorted(set(values)))


# ============================================================
# BUILD ONE ROW PER SNP
# ============================================================

rows = []

for variant in vep:

    rsid = variant.get("id")

    row = {
        "RSID": rsid,

        # Basic variant information
        "Assembly": variant.get("assembly_name"),
        "CHR_GRCh38": variant.get("seq_region_name"),
        "POS_GRCh38": variant.get("start"),
        "Allele_string": variant.get("allele_string"),

        # Overall consequence
        "Most_severe_consequence":
            variant.get("most_severe_consequence"),

        # Gene annotation
        "Gene_symbols": "",
        "Ensembl_gene_IDs": "",
        "Transcript_IDs": "",

        # Functional annotation
        "Consequence_terms": "",
        "Biotypes": "",
        "Impacts": "",

        # Transcript information
        "Canonical_transcripts": "",
        "MANE_transcripts": "",

        # Coding information
        "Amino_acids": "",
        "Codons": "",
        "Protein_changes": "",

        # Prediction scores
        "SIFT": "",
        "PolyPhen": "",
        "CADD": "",
        "AlphaMissense": "",

        # gnomAD / population frequencies
        "gnomADg_AF": "",
        "gnomADg_AFR": "",
        "gnomADg_AMR": "",
        "gnomADg_EAS": "",
        "gnomADg_NFE": "",
        "gnomADg_SAS": "",
        "gnomADg_FIN": "",
        "gnomADg_ASJ": "",
        "gnomADg_AMI": "",
        "gnomADg_MID": "",
        "gnomADg_REMAINING": "",

        # Other population frequencies returned by VEP
        "AF": "",
        "AFR": "",
        "AMR": "",
        "EAS": "",
        "EUR": "",
        "SAS": "",

        # Colocated variants
        "Colocated_variants": ""
    }


    # ========================================================
    # TRANSCRIPT CONSEQUENCES
    # ========================================================

    transcript_consequences = variant.get(
        "transcript_consequences",
        []
    )

    genes = []
    gene_ids = []
    transcripts = []
    consequences = []
    biotypes = []
    impacts = []

    canonical = []
    mane = []

    amino_acids = []
    codons = []
    protein_changes = []

    sift = []
    polyphen = []
    cadd = []
    alphamissense = []


    for tc in transcript_consequences:

        # Gene
        if tc.get("gene_symbol"):
            genes.append(tc["gene_symbol"])

        if tc.get("gene_id"):
            gene_ids.append(tc["gene_id"])

        # Transcript
        if tc.get("transcript_id"):
            transcripts.append(tc["transcript_id"])

        # Consequences
        for consequence in tc.get(
            "consequence_terms",
            []
        ):
            consequences.append(consequence)

        # Biotype
        if tc.get("biotype"):
            biotypes.append(tc["biotype"])

        # Impact
        if tc.get("impact"):
            impacts.append(tc["impact"])

        # Canonical transcript
        if tc.get("canonical"):
            canonical.append(
                tc.get("transcript_id")
            )

        # MANE transcript
        if tc.get("mane_select"):
            mane.append(
                tc.get("mane_select")
            )

        # Amino acids
        if tc.get("amino_acids"):
            amino_acids.append(
                tc["amino_acids"]
            )

        # Codons
        if tc.get("codons"):
            codons.append(
                tc["codons"]
            )

        # Protein change
        if (
            tc.get("protein_start") is not None
            and tc.get("amino_acids")
        ):
            protein_changes.append(
                f"{tc.get('amino_acids')}"
                f"{tc.get('protein_start')}"
            )

        # SIFT
        if tc.get("sift_prediction"):

            sift.append(
                f"{tc.get('sift_prediction')}"
                f"({tc.get('sift_score')})"
            )

        # PolyPhen
        if tc.get("polyphen_prediction"):

            polyphen.append(
                f"{tc.get('polyphen_prediction')}"
                f"({tc.get('polyphen_score')})"
            )

        # CADD
        if tc.get("cadd_phred") is not None:

            cadd.append(
                str(tc["cadd_phred"])
            )

        # AlphaMissense
        if tc.get("alphamissense_prediction"):

            alphamissense.append(
                str(tc["alphamissense_prediction"])
            )


    # Put transcript annotations into the row

    row["Gene_symbols"] = unique_join(genes)

    row["Ensembl_gene_IDs"] = unique_join(
        gene_ids
    )

    row["Transcript_IDs"] = unique_join(
        transcripts
    )

    row["Consequence_terms"] = unique_join(
        consequences
    )

    row["Biotypes"] = unique_join(
        biotypes
    )

    row["Impacts"] = unique_join(
        impacts
    )

    row["Canonical_transcripts"] = unique_join(
        canonical
    )

    row["MANE_transcripts"] = unique_join(
        mane
    )

    row["Amino_acids"] = unique_join(
        amino_acids
    )

    row["Codons"] = unique_join(
        codons
    )

    row["Protein_changes"] = unique_join(
        protein_changes
    )

    row["SIFT"] = unique_join(sift)

    row["PolyPhen"] = unique_join(polyphen)

    row["CADD"] = unique_join(cadd)
a
    row["AlphaMissense"] = unique_join(
        alphamissense
    )


    # ========================================================
    # INTERGENIC CONSEQUENCES
    # ========================================================

    intergenic = variant.get(
        "intergenic_consequences",
        []
    )

    for ic in intergenic:

        for consequence in ic.get(
            "consequence_terms",
            []
        ):
            consequences.append(consequence)

        if ic.get("impact"):
            impacts.append(ic["impact"])

    # Update consequence field
    row["Consequence_terms"] = unique_join(
        consequences
    )

    row["Impacts"] = unique_join(
        impacts
    )


    # ========================================================
    # GNOMAD / POPULATION FREQUENCIES
    # ========================================================

    colocated = variant.get(
        "colocated_variants",
        []
    )

    colocated_ids = []

    for cv in colocated:

        if cv.get("id"):
            colocated_ids.append(
                cv["id"]
            )

        frequencies = cv.get(
            "frequencies",
            {}
        )

        # Frequencies are allele-specific
        for allele, freq_data in frequencies.items():

            if not isinstance(freq_data, dict):
                continue

            # gnomAD Genome
            frequency_map = {

                "gnomadg":
                    "gnomADg_AF",

                "gnomadg_afr":
                    "gnomADg_AFR",

                "gnomadg_amr":
                    "gnomADg_AMR",

                "gnomadg_eas":
                    "gnomADg_EAS",

                "gnomadg_nfe":
                    "gnomADg_NFE",

                "gnomadg_sas":
                    "gnomADg_SAS",

                "gnomadg_fin":
                    "gnomADg_FIN",

                "gnomadg_asj":
                    "gnomADg_ASJ",

                "gnomadg_ami":
                    "gnomADg_AMI",

                "gnomadg_mid":
                    "gnomADg_MID",

                "gnomadg_remaining":
                    "gnomADg_REMAINING",

                # Other AF fields
                "af":
                    "AF",

                "afr":
                    "AFR",

                "amr":
                    "AMR",

                "eas":
                    "EAS",

                "eur":
                    "EUR",

                "sas":
                    "SAS"
            }

            for source, destination in frequency_map.items():

                if source in freq_data:

                    row[destination] = freq_data[
                        source
                    ]


    row["Colocated_variants"] = unique_join(
        colocated_ids
    )


    rows.append(row)


# ============================================================
# CREATE DATAFRAME
# ============================================================

annotation_df = pd.DataFrame(rows)


# ============================================================
# SAVE
# ============================================================

annotation_df.to_csv(
    OUTPUT,
    sep="\t",
    index=False
)

print("\nFinished!")
print(
    f"Saved {len(annotation_df)} variants to:"
)
print(OUTPUT)

# ============================================================
# PRINT IMPORTANT COLUMNS
# ============================================================

print("\nSummary:")
print(
    annotation_df[
        [
            "RSID",
            "CHR_GRCh38",
            "POS_GRCh38",
            "Gene_symbols",
            "Most_severe_consequence",
            "gnomADg_AF"
        ]
    ].to_string(index=False)
)

In [7]:
import json
import pandas as pd


# ============================================================
# INPUT / OUTPUT FILES
# ============================================================

VEP_FILE = "top39_VEP_raw.json"
GWAS_FILE = "top39_for_annotation.tsv"

OUTPUT_FILE = "top39_VEP_annotation.tsv"


# ============================================================
# LOAD ORIGINAL GWAS DATA
# ============================================================

gwas = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    header=None,
    names=[
        "CHR",
        "POS",
        "RSID",
        "GWAS_REF",
        "GWAS_ALT",
        "Y",
        "A1"
    ],
    dtype={
        "CHR": str,
        "RSID": str,
        "GWAS_REF": str,
        "GWAS_ALT": str,
        "Y": str,
        "A1": str
    }
)

gwas["POS"] = pd.to_numeric(
    gwas["POS"],
    errors="coerce"
)

print(f"Loaded {len(gwas)} GWAS variants")


# ============================================================
# LOAD VEP JSON
# ============================================================

with open(VEP_FILE, "r") as f:
    vep = json.load(f)

print(f"Loaded {len(vep)} VEP variants")


# ============================================================
# CREATE GWAS LOOKUP
# ============================================================

gwas_lookup = (
    gwas
    .set_index("RSID")
    .to_dict("index")
)


# ============================================================
# HELPER FUNCTION
# ============================================================

def unique_join(values):

    values = [
        str(x)
        for x in values
        if x is not None
        and str(x) not in ["", "nan", "None"]
    ]

    return ";".join(sorted(set(values)))


# ============================================================
# CREATE OUTPUT ROWS
# ============================================================

rows = []


for variant in vep:

    rsid = variant.get("id")


    # ========================================================
    # ORIGINAL GWAS INFORMATION
    # ========================================================

    if rsid in gwas_lookup:

        gwas_info = gwas_lookup[rsid]

        gw_chr = gwas_info["CHR"]
        gw_pos = gwas_info["POS"]
        gw_ref = gwas_info["GWAS_REF"]
        gw_alt = gwas_info["GWAS_ALT"]
        gw_a1 = gwas_info["A1"]

    else:

        gw_chr = ""
        gw_pos = ""
        gw_ref = ""
        gw_alt = ""
        gw_a1 = ""


    # ========================================================
    # BASIC VEP INFORMATION
    # ========================================================

    row = {

        # Original GWAS information
        "RSID": rsid,
        "GWAS_CHR": gw_chr,
        "GWAS_POS": gw_pos,
        "GWAS_REF": gw_ref,
        "GWAS_ALT": gw_alt,
        "GWAS_A1": gw_a1,

        # VEP information
        "VEP_Assembly":
            variant.get("assembly_name", ""),

        "VEP_CHR":
            variant.get("seq_region_name", ""),

        "VEP_POS":
            variant.get("start", ""),

        "VEP_Allele_string":
            variant.get("allele_string", ""),

        "Most_severe_consequence":
            variant.get(
                "most_severe_consequence",
                ""
            ),

        # Gene annotation
        "Gene_symbols": "",
        "Ensembl_gene_IDs": "",
        "Transcript_IDs": "",
        "Consequence_terms": "",
        "Biotypes": "",
        "Impacts": "",

        # Transcript information
        "Canonical_transcripts": "",
        "MANE_transcripts": "",

        # Coding information
        "Amino_acids": "",
        "Codons": "",
        "Protein_changes": "",

        # Prediction scores
        "SIFT": "",
        "PolyPhen": "",
        "CADD": "",
        "AlphaMissense": "",

        # gnomAD
        "gnomADg_Allele": "",
        "gnomADg_AF": "",
        "gnomADg_AF_Matches_GWAS_ALT": "",

        "gnomADg_AFR": "",
        "gnomADg_AMR": "",
        "gnomADg_EAS": "",
        "gnomADg_NFE": "",
        "gnomADg_SAS": "",
        "gnomADg_FIN": "",
        "gnomADg_ASJ": "",
        "gnomADg_AMI": "",
        "gnomADg_MID": "",
        "gnomADg_REMAINING": "",

        # Other VEP population frequencies
        "AF": "",
        "AFR": "",
        "AMR": "",
        "EAS": "",
        "EUR": "",
        "SAS": "",

        # Other known variants
        "Colocated_variants": ""
    }


    # ========================================================
    # TRANSCRIPT CONSEQUENCES
    # ========================================================

    transcript_consequences = variant.get(
        "transcript_consequences",
        []
    )

    genes = []
    gene_ids = []
    transcripts = []
    consequences = []
    biotypes = []
    impacts = []

    canonical = []
    mane = []

    amino_acids = []
    codons = []
    protein_changes = []

    sift = []
    polyphen = []
    cadd = []
    alphamissense = []


    # --------------------------------------------------------
    # IMPORTANT:
    # Only use transcript consequences for GWAS_ALT
    # --------------------------------------------------------

    for tc in transcript_consequences:

        variant_allele = tc.get(
            "variant_allele"
        )


        # If GWAS ALT is available,
        # require exact allele match

        if gw_alt != "":

            if variant_allele != gw_alt:
                continue


        # ----------------------------------------------------
        # Gene
        # ----------------------------------------------------

        if tc.get("gene_symbol"):

            genes.append(
                tc["gene_symbol"]
            )


        if tc.get("gene_id"):

            gene_ids.append(
                tc["gene_id"]
            )


        # ----------------------------------------------------
        # Transcript
        # ----------------------------------------------------

        if tc.get("transcript_id"):

            transcripts.append(
                tc["transcript_id"]
            )


        # ----------------------------------------------------
        # Consequence
        # ----------------------------------------------------

        for consequence in tc.get(
            "consequence_terms",
            []
        ):

            consequences.append(
                consequence
            )


        # ----------------------------------------------------
        # Biotype
        # ----------------------------------------------------

        if tc.get("biotype"):

            biotypes.append(
                tc["biotype"]
            )


        # ----------------------------------------------------
        # Impact
        # ----------------------------------------------------

        if tc.get("impact"):

            impacts.append(
                tc["impact"]
            )


        # ----------------------------------------------------
        # Canonical transcript
        # ----------------------------------------------------

        if tc.get("canonical"):

            canonical.append(
                tc.get("transcript_id")
            )


        # ----------------------------------------------------
        # MANE transcript
        # ----------------------------------------------------

        if tc.get("mane_select"):

            mane.append(
                tc.get("mane_select")
            )


        # ----------------------------------------------------
        # Amino acid change
        # ----------------------------------------------------

        if tc.get("amino_acids"):

            amino_acids.append(
                tc["amino_acids"]
            )


        # ----------------------------------------------------
        # Codon change
        # ----------------------------------------------------

        if tc.get("codons"):

            codons.append(
                tc["codons"]
            )


        # ----------------------------------------------------
        # Protein position
        # ----------------------------------------------------

        if (
            tc.get("protein_start") is not None
            and tc.get("amino_acids")
        ):

            protein_changes.append(
                f"{tc.get('amino_acids')}"
                f"{tc.get('protein_start')}"
            )


        # ----------------------------------------------------
        # SIFT
        # ----------------------------------------------------

        if tc.get("sift_prediction"):

            sift.append(
                f"{tc.get('sift_prediction')}"
                f"({tc.get('sift_score')})"
            )


        # ----------------------------------------------------
        # PolyPhen
        # ----------------------------------------------------

        if tc.get("polyphen_prediction"):

            polyphen.append(
                f"{tc.get('polyphen_prediction')}"
                f"({tc.get('polyphen_score')})"
            )


        # ----------------------------------------------------
        # CADD
        # ----------------------------------------------------

        if tc.get("cadd_phred") is not None:

            cadd.append(
                str(tc["cadd_phred"])
            )


        # ----------------------------------------------------
        # AlphaMissense
        # ----------------------------------------------------

        if tc.get("alphamissense_prediction"):

            alphamissense.append(
                str(
                    tc[
                        "alphamissense_prediction"
                    ]
                )
            )


    # ========================================================
    # STORE TRANSCRIPT ANNOTATIONS
    # ========================================================

    row["Gene_symbols"] = unique_join(
        genes
    )

    row["Ensembl_gene_IDs"] = unique_join(
        gene_ids
    )

    row["Transcript_IDs"] = unique_join(
        transcripts
    )

    row["Consequence_terms"] = unique_join(
        consequences
    )

    row["Biotypes"] = unique_join(
        biotypes
    )

    row["Impacts"] = unique_join(
        impacts
    )

    row["Canonical_transcripts"] = unique_join(
        canonical
    )

    row["MANE_transcripts"] = unique_join(
        mane
    )

    row["Amino_acids"] = unique_join(
        amino_acids
    )

    row["Codons"] = unique_join(
        codons
    )

    row["Protein_changes"] = unique_join(
        protein_changes
    )

    row["SIFT"] = unique_join(
        sift
    )

    row["PolyPhen"] = unique_join(
        polyphen
    )

    row["CADD"] = unique_join(
        cadd
    )

    row["AlphaMissense"] = unique_join(
        alphamissense
    )


    # ========================================================
    # COLOCATED VARIANTS + POPULATION FREQUENCIES
    # ========================================================

    colocated = variant.get(
        "colocated_variants",
        []
    )

    colocated_ids = []


    # We will store the gnomAD allele
    # specifically for GWAS_ALT

    gnomad_allele = ""
    gnomad_af = ""


    for cv in colocated:


        # ----------------------------------------------------
        # Colocated variant IDs
        # ----------------------------------------------------

        if cv.get("id"):

            colocated_ids.append(
                cv["id"]
            )


        # ----------------------------------------------------
        # Frequency information
        # ----------------------------------------------------

        frequencies = cv.get(
            "frequencies",
            {}
        )


        for allele, freq_data in frequencies.items():

            if not isinstance(
                freq_data,
                dict
            ):

                continue


            # ------------------------------------------------
            # ONLY USE YOUR GWAS ALT
            # ------------------------------------------------

            if gw_alt != "":

                if allele != gw_alt:
                    continue


            # ------------------------------------------------
            # GNOMAD OVERALL FREQUENCY
            # ------------------------------------------------

            if "gnomadg" in freq_data:

                gnomad_allele = allele

                gnomad_af = (
                    freq_data["gnomadg"]
                )


            # ------------------------------------------------
            # GNOMAD POPULATION FREQUENCIES
            # ------------------------------------------------

            if "gnomadg_afr" in freq_data:

                row["gnomADg_AFR"] = (
                    freq_data["gnomadg_afr"]
                )


            if "gnomadg_amr" in freq_data:

                row["gnomADg_AMR"] = (
                    freq_data["gnomadg_amr"]
                )


            if "gnomadg_eas" in freq_data:

                row["gnomADg_EAS"] = (
                    freq_data["gnomadg_eas"]
                )


            if "gnomadg_nfe" in freq_data:

                row["gnomADg_NFE"] = (
                    freq_data["gnomadg_nfe"]
                )


            if "gnomadg_sas" in freq_data:

                row["gnomADg_SAS"] = (
                    freq_data["gnomadg_sas"]
                )


            if "gnomadg_fin" in freq_data:

                row["gnomADg_FIN"] = (
                    freq_data["gnomadg_fin"]
                )


            if "gnomadg_asj" in freq_data:

                row["gnomADg_ASJ"] = (
                    freq_data["gnomadg_asj"]
                )


            if "gnomadg_ami" in freq_data:

                row["gnomADg_AMI"] = (
                    freq_data["gnomadg_ami"]
                )


            if "gnomadg_mid" in freq_data:

                row["gnomADg_MID"] = (
                    freq_data["gnomadg_mid"]
                )


            if "gnomadg_remaining" in freq_data:

                row["gnomADg_REMAINING"] = (
                    freq_data[
                        "gnomadg_remaining"
                    ]
                )


            # ------------------------------------------------
            # OTHER VEP FREQUENCIES
            # ------------------------------------------------

            if "af" in freq_data:

                row["AF"] = (
                    freq_data["af"]
                )


            if "afr" in freq_data:

                row["AFR"] = (
                    freq_data["afr"]
                )


            if "amr" in freq_data:

                row["AMR"] = (
                    freq_data["amr"]
                )


            if "eas" in freq_data:

                row["EAS"] = (
                    freq_data["eas"]
                )


            if "eur" in freq_data:

                row["EUR"] = (
                    freq_data["eur"]
                )


            if "sas" in freq_data:

                row["SAS"] = (
                    freq_data["sas"]
                )


    # ========================================================
    # STORE GNOMAD ALLELE + MATCH FLAG
    # ========================================================

    row["gnomADg_Allele"] = (
        gnomad_allele
    )

    row["gnomADg_AF"] = (
        gnomad_af
    )


    if gnomad_allele != "":

        row[
            "gnomADg_AF_Matches_GWAS_ALT"
        ] = (
            gnomad_allele == gw_alt
        )

    else:

        row[
            "gnomADg_AF_Matches_GWAS_ALT"
        ] = ""


    # ========================================================
    # COLOCATED VARIANT IDs
    # ========================================================

    row["Colocated_variants"] = unique_join(
        colocated_ids
    )


    # ========================================================
    # ADD ROW
    # ========================================================

    rows.append(row)


# ============================================================
# CREATE DATAFRAME
# ============================================================

annotation_df = pd.DataFrame(
    rows
)


# ============================================================
# CHECK WHETHER VEP LOCATION MATCHES GWAS LOCATION
# ============================================================

annotation_df["Location_match"] = (
    annotation_df["GWAS_CHR"].astype(str)
    ==
    annotation_df["VEP_CHR"].astype(str)
) & (
    pd.to_numeric(
        annotation_df["GWAS_POS"],
        errors="coerce"
    )
    ==
    pd.to_numeric(
        annotation_df["VEP_POS"],
        errors="coerce"
    )
)


# ============================================================
# SAVE
# ============================================================

annotation_df.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("\n========================================")
print("FINISHED")
print("========================================")

print(
    f"\nSaved {len(annotation_df)} variants to:"
)

print(
    OUTPUT_FILE
)


# ============================================================
# LOCATION CHECK
# ============================================================

print("\n========================================")
print("LOCATION MATCHES")
print("========================================")

print(
    annotation_df[
        "Location_match"
    ].value_counts()
)


# ============================================================
# GNOMAD ALLELE CHECK
# ============================================================

print("\n========================================")
print("GNOMAD ALLELE CHECK")
print("========================================")

print(
    annotation_df[
        [
            "RSID",
            "GWAS_REF",
            "GWAS_ALT",
            "VEP_Allele_string",
            "gnomADg_Allele",
            "gnomADg_AF",
            "gnomADg_AF_Matches_GWAS_ALT"
        ]
    ].to_string(index=False)
)


# ============================================================
# SNPs WHERE GNOMAD ALLELE DOES NOT MATCH
# ============================================================

mismatch = annotation_df[
    annotation_df[
        "gnomADg_AF_Matches_GWAS_ALT"
    ] == False
]


if len(mismatch) > 0:

    print("\n========================================")
    print("GNOMAD ALLELE MISMATCHES")
    print("========================================")

    print(
        mismatch[
            [
                "RSID",
                "GWAS_REF",
                "GWAS_ALT",
                "gnomADg_Allele",
                "gnomADg_AF"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "\nAll available gnomAD frequencies "
        "match GWAS_ALT."
    )

Loaded 39 GWAS variants
Loaded 46 VEP variants

FINISHED

Saved 46 variants to:
top39_VEP_annotation.tsv

LOCATION MATCHES
Location_match
True     39
False     7
Name: count, dtype: int64

GNOMAD ALLELE CHECK
      RSID GWAS_REF GWAS_ALT VEP_Allele_string gnomADg_Allele gnomADg_AF gnomADg_AF_Matches_GWAS_ALT
 rs1560036        G        A               G/A              A     0.4109                        True
rs13081814        A        G               A/G              G     0.1525                        True
 rs4923807        T        C             C/A/T                                                      
rs11818063        C        T             C/G/T              T      0.256                        True
rs62121092        G        A             G/A/T              A     0.1244                        True
 rs6441370        T        C           C/A/G/T                                                      
 rs3740779        G        A             A/G/T                                      

In [8]:
# adding mafs (all, cases, controls)
import pandas as pd

# ============================================================
# INPUT FILES
# ============================================================

vep_file = "top39_VEP_annotation.tsv"

sample_maf_file = "d13_freq.frq"
case_maf_file = "d13_case.frq"
control_maf_file = "d13_control.frq"

output_file = "top39_vep_annotation_with_mafs.tsv"


# ============================================================
# READ VEP ANNOTATION
# ============================================================

vep = pd.read_csv(
    vep_file,
    sep="\t",
    dtype={"RSID": str}
)

print("VEP variants:", len(vep))


# ============================================================
# FUNCTION TO READ PLINK .FRQ FILE
# ============================================================

def read_maf_file(filename, maf_column_name):

    df = pd.read_csv(
        filename,
        sep=r"\s+",
        dtype={"SNP": str}
    )

    # Keep only SNP ID and MAF
    df = df[["SNP", "MAF"]].copy()

    # Rename columns
    df = df.rename(
        columns={
            "SNP": "RSID",
            "MAF": maf_column_name
        }
    )

    return df


# ============================================================
# READ ALL THREE MAF FILES
# ============================================================

sample_maf = read_maf_file(
    sample_maf_file,
    "Sample_MAF"
)

case_maf = read_maf_file(
    case_maf_file,
    "Case_MAF"
)

control_maf = read_maf_file(
    control_maf_file,
    "Control_MAF"
)


print("Sample MAF variants:", len(sample_maf))
print("Case MAF variants:", len(case_maf))
print("Control MAF variants:", len(control_maf))


# ============================================================
# MERGE MAFs INTO VEP TABLE
# ============================================================

result = vep.merge(
    sample_maf,
    on="RSID",
    how="left"
)

result = result.merge(
    case_maf,
    on="RSID",
    how="left"
)

result = result.merge(
    control_maf,
    on="RSID",
    how="left"
)


# ============================================================
# CHECK MATCHING
# ============================================================

print("\nMAF matching:")

print(
    "Sample MAF:",
    result["Sample_MAF"].notna().sum(),
    "/",
    len(result)
)

print(
    "Case MAF:",
    result["Case_MAF"].notna().sum(),
    "/",
    len(result)
)

print(
    "Control MAF:",
    result["Control_MAF"].notna().sum(),
    "/",
    len(result)
)


# ============================================================
# REPORT MISSING VALUES
# ============================================================

missing = result[
    result[
        [
            "Sample_MAF",
            "Case_MAF",
            "Control_MAF"
        ]
    ].isna().any(axis=1)
]

if len(missing) > 0:

    print("\nSNPs with missing MAF values:")

    print(
        missing[
            [
                "RSID",
                "Sample_MAF",
                "Case_MAF",
                "Control_MAF"
            ]
        ].to_string(index=False)
    )


# ============================================================
# SAVE
# ============================================================

result.to_csv(
    output_file,
    sep="\t",
    index=False
)

print(
    f"\nSaved to: {output_file}"
)

VEP variants: 46
Sample MAF variants: 181595
Case MAF variants: 181595
Control MAF variants: 181595

MAF matching:
Sample MAF: 46 / 46
Case MAF: 46 / 46
Control MAF: 46 / 46

Saved to: top39_vep_annotation_with_mafs.tsv


I removed 6 rows with False in Location_match manually. The gnomad alt allele matches in all 17 variants for which frequencies are available

In [9]:
# adding GWAS catalogue info:
import requests
import pandas as pd
import time
from tqdm import tqdm


# ============================================================
# INPUT / OUTPUT
# ============================================================

INPUT_FILE = "top39_VEP_annotation.tsv"
OUTPUT_FILE = "top39_VEP_annotation_TOPMed_GWASCatalog.tsv"


# ============================================================
# LOAD YOUR EXISTING VEP TABLE
# ============================================================

df = pd.read_csv(
    INPUT_FILE,
    sep="\t",
    dtype={"RSID": str}
)

print(f"Loaded {len(df)} variants")


# ============================================================
# INITIALIZE NEW COLUMNS
# ============================================================

new_columns = {

    # TOPMed
    "TOPMed_Allele": "",
    "TOPMed_AF": "",
    "TOPMed_Matches_GWAS_ALT": "",

    # GWAS Catalog
    "GWAS_Catalog_Association": "",
    "GWAS_Catalog_Traits": "",
    "GWAS_Catalog_P": "",
    "GWAS_Catalog_Study": "",
    "GWAS_Catalog_PubMed": "",
    "GWAS_Catalog_Effect_Allele": ""
}

for column in new_columns:
    df[column] = ""


# ============================================================
# ENSEMBL VARIATION API
#
# Used for TOPMed
#
# Example:
# https://rest.ensembl.org/variation/human/rs56116432
# ============================================================

ENSEMBL_URL = (
    "https://rest.ensembl.org/variation/human/{}"
)


def get_topmed(rsid, gw_alt):

    url = ENSEMBL_URL.format(rsid)

    headers = {
        "Content-Type": "application/json"
    }

    params = {
        "pops": "1"
    }

    try:

        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30
        )

        if response.status_code != 200:

            print(
                f"  TOPMed failed for {rsid}: "
                f"HTTP {response.status_code}"
            )

            return "", "", ""


        data = response.json()


        # ----------------------------------------------------
        # Look through population allele frequencies
        # ----------------------------------------------------

        topmed_results = []


        for pop in data.get(
            "populations",
            []
        ):

            # Population/source names vary in Ensembl
            source = str(
                pop.get("population", "")
            )

            source_upper = source.upper()


            if "TOPMED" not in source_upper:
                continue


            allele = pop.get(
                "allele",
                ""
            )

            frequency = pop.get(
                "frequency",
                None
            )


            if frequency is None:
                continue


            topmed_results.append(
                (
                    allele,
                    frequency
                )
            )


        # ----------------------------------------------------
        # If no TOPMed result was found
        # ----------------------------------------------------

        if len(topmed_results) == 0:

            return "", "", ""


        # ----------------------------------------------------
        # Find the GWAS ALT allele
        # ----------------------------------------------------

        matching = [
            x
            for x in topmed_results
            if x[0] == gw_alt
        ]


        if len(matching) > 0:

            allele = matching[0][0]
            frequency = matching[0][1]

            return (
                allele,
                frequency,
                True
            )


        # ----------------------------------------------------
        # TOPMed exists, but not for GWAS ALT
        # ----------------------------------------------------

        alleles = ";".join(
            sorted(
                set(
                    x[0]
                    for x in topmed_results
                )
            )
        )

        return (
            alleles,
            "",
            False
        )


    except Exception as e:

        print(
            f"  TOPMed error for {rsid}: {e}"
        )

        return "", "", ""


# ============================================================
# GWAS CATALOG API V2
#
# Endpoint:
# /v2/associations
#
# Filter:
# rs_id
# ============================================================

GWAS_URL = (
    "https://www.ebi.ac.uk/gwas/rest/api/v2/associations"
)


def get_gwas_catalog(rsid):

    params = {
        "rs_id": rsid,
        "page": 0,
        "size": 100
    }

    headers = {
        "Accept": "application/json"
    }


    try:

        response = requests.get(
            GWAS_URL,
            params=params,
            headers=headers,
            timeout=30
        )


        if response.status_code != 200:

            print(
                f"  GWAS Catalog failed for "
                f"{rsid}: HTTP "
                f"{response.status_code}"
            )

            return []


        data = response.json()


        # GWAS Catalog v2 is paginated
        # Extract embedded association records

        embedded = data.get(
            "_embedded",
            {}
        )

        associations = embedded.get(
            "associations",
            []
        )


        return associations


    except Exception as e:

        print(
            f"  GWAS Catalog error for "
            f"{rsid}: {e}"
        )

        return []


# ============================================================
# HELPER FOR GWAS CATALOG VALUES
# ============================================================

def get_value(obj, *keys):

    for key in keys:

        if isinstance(obj, dict):

            value = obj.get(key)

            if value is not None:
                return value

    return ""


# ============================================================
# PROCESS ALL 39 SNPs
# ============================================================

for i in tqdm(
    range(len(df)),
    desc="Annotating TOPMed + GWAS Catalog"
):

    rsid = df.loc[i, "RSID"]

    gw_alt = str(
        df.loc[i, "GWAS_ALT"]
    )


    # ========================================================
    # TOPMed
    # ========================================================

    allele, af, match = get_topmed(
        rsid,
        gw_alt
    )


    df.loc[
        i,
        "TOPMed_Allele"
    ] = allele

    df.loc[
        i,
        "TOPMed_AF"
    ] = af

    df.loc[
        i,
        "TOPMed_Matches_GWAS_ALT"
    ] = match


    # ========================================================
    # GWAS CATALOG
    # ========================================================

    associations = get_gwas_catalog(
        rsid
    )


    if len(associations) == 0:

        df.loc[
            i,
            "GWAS_Catalog_Association"
        ] = "No"

    else:

        df.loc[
            i,
            "GWAS_Catalog_Association"
        ] = "Yes"


        traits = []
        pvalues = []
        studies = []
        pubmeds = []
        effect_alleles = []


        # ----------------------------------------------------
        # Extract information from every association
        # ----------------------------------------------------

        for association in associations:


            # ------------------------------------------------
            # Trait
            # ------------------------------------------------

            trait = get_value(
                association,
                "trait",
                "efo_trait",
                "reported_trait"
            )

            if trait:
                traits.append(
                    str(trait)
                )


            # ------------------------------------------------
            # P value
            # ------------------------------------------------

            pvalue = get_value(
                association,
                "p_value",
                "pvalue"
            )

            if pvalue:
                pvalues.append(
                    str(pvalue)
                )


            # ------------------------------------------------
            # Study
            # ------------------------------------------------

            study = get_value(
                association,
                "study_accession",
                "accession_id",
                "study"
            )

            if study:
                studies.append(
                    str(study)
                )


            # ------------------------------------------------
            # PubMed
            # ------------------------------------------------

            pubmed = get_value(
                association,
                "pubmed_id",
                "pubmed"
            )

            if pubmed:
                pubmeds.append(
                    str(pubmed)
                )


            # ------------------------------------------------
            # Effect/risk allele
            # ------------------------------------------------

            effect_allele = get_value(
                association,
                "effect_allele",
                "risk_allele"
            )

            if effect_allele:
                effect_alleles.append(
                    str(effect_allele)
                )


        # ----------------------------------------------------
        # Remove duplicates
        # ----------------------------------------------------

        traits = sorted(
            set(traits)
        )

        pvalues = sorted(
            set(pvalues)
        )

        studies = sorted(
            set(studies)
        )

        pubmeds = sorted(
            set(pubmeds)
        )

        effect_alleles = sorted(
            set(effect_alleles)
        )


        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        df.loc[
            i,
            "GWAS_Catalog_Traits"
        ] = ";".join(traits)

        df.loc[
            i,
            "GWAS_Catalog_P"
        ] = ";".join(pvalues)

        df.loc[
            i,
            "GWAS_Catalog_Study"
        ] = ";".join(studies)

        df.loc[
            i,
            "GWAS_Catalog_PubMed"
        ] = ";".join(pubmeds)

        df.loc[
            i,
            "GWAS_Catalog_Effect_Allele"
        ] = ";".join(effect_alleles)


    # --------------------------------------------------------
    # Be polite to APIs
    # --------------------------------------------------------

    time.sleep(0.1)


# ============================================================
# SAVE
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False
)


print()
print("=" * 60)
print("FINISHED")
print("=" * 60)

print(
    f"Saved to:\n{OUTPUT_FILE}"
)


# ============================================================
# SUMMARY
# ============================================================

print("\nTOPMed results:")

print(
    df[
        [
            "RSID",
            "GWAS_ALT",
            "TOPMed_Allele",
            "TOPMed_AF",
            "TOPMed_Matches_GWAS_ALT"
        ]
    ].to_string(index=False)
)


print("\nGWAS Catalog hits:")

hits = df[
    df["GWAS_Catalog_Association"]
    == "Yes"
]

print(
    hits[
        [
            "RSID",
            "GWAS_ALT",
            "GWAS_Catalog_Traits",
            "GWAS_Catalog_P",
            "GWAS_Catalog_Study"
        ]
    ].to_string(index=False)
)

print(
    f"\nGWAS Catalog matches: "
    f"{len(hits)}/{len(df)}"
)

Loaded 46 variants


Annotating TOPMed + GWAS Catalog:   7%|██▋                                       | 3/46 [00:59<14:46, 20.62s/it]

  TOPMed failed for rs11818063: HTTP 503


Annotating TOPMed + GWAS Catalog:  13%|█████▍                                    | 6/46 [02:13<15:41, 23.55s/it]

  TOPMed failed for rs3740779: HTTP 500


...

  TOPMed failed for rs9263600: HTTP 503


Annotating TOPMed + GWAS Catalog:  98%|████████████████████████████████████████ | 45/46 [09:51<00:12, 12.79s/it]

  TOPMed failed for rs9853234: HTTP 503


Annotating TOPMed + GWAS Catalog: 100%|█████████████████████████████████████████| 46/46 [10:12<00:00, 13.32s/it]


FINISHED
Saved to:
top39_VEP_annotation_TOPMed_GWASCatalog.tsv

TOPMed results:
      RSID GWAS_ALT TOPMed_Allele TOPMed_AF TOPMed_Matches_GWAS_ALT
 rs1560036        A             A  0.414564                    True
rs13081814        G             G  0.157397                    True
 rs4923807        C             C   0.44814                    True
rs11818063        T                                                
rs62121092        A             A  0.129667                    True
 rs6441370        C             C  0.323187                    True
 rs3740779        A                                                
  rs835259        T                                                
 rs4470535        G                                                
 rs1885152        T                                                
 rs1265425        C                                                
 rs4330304        A                                                
 rs3795179        G             G  

In [ ]:
# getting nearest gene:
import requests
import pandas as pd
import time
from tqdm import tqdm


# ============================================================
# INPUT / OUTPUT
# ============================================================

TOP39_FILE = "top39.txt"

OUTPUT = "top39_nearest_gene.tsv"


# ============================================================
# SETTINGS
# ============================================================

# Search up to 1 Mb for nearby genes
DISTANCE = 1_000_000

VEP_URL = "https://rest.ensembl.org/vep/human/region"

HEADERS = {
    "Accept": "application/json"
}


# ============================================================
# LOAD TOP39
# ============================================================

# Your file:
#
# CHR POS RSID REF ALT ...
#
top39 = pd.read_csv(
    TOP39_FILE,
    sep=r"\s+",
    header=None,
    dtype=str
)

# Keep first five columns only
top39 = top39.iloc[:, :5]

top39.columns = [
    "CHR",
    "POS",
    "RSID",
    "REF",
    "ALT"
]

print(
    f"Loaded {len(top39)} variants"
)


# ============================================================
# CHECK FOR DUPLICATES
# ============================================================

if top39["RSID"].duplicated().any():

    print(
        "\nWARNING: duplicate RSIDs found:"
    )

    print(
        top39[
            top39["RSID"].duplicated(
                keep=False
            )
        ]["RSID"].tolist()
    )


# ============================================================
# CREATE OUTPUT COLUMNS
# ============================================================

top39["Nearest_gene"] = pd.Series(
    pd.NA,
    index=top39.index,
    dtype="string"
)

top39["Nearest_gene_ID"] = pd.Series(
    pd.NA,
    index=top39.index,
    dtype="string"
)

top39["Nearest_gene_distance_bp"] = pd.Series(
    pd.NA,
    index=top39.index,
    dtype="Int64"
)

top39["Nearest_gene_consequence"] = pd.Series(
    pd.NA,
    index=top39.index,
    dtype="string"
)

top39["Nearby_genes"] = pd.Series(
    pd.NA,
    index=top39.index,
    dtype="string"
)

top39["Nearby_gene_distances_bp"] = pd.Series(
    pd.NA,
    index=top39.index,
    dtype="string"
)


# ============================================================
# VEP QUERY FUNCTION
# ============================================================

def query_vep(
    chromosome,
    position,
    alt,
    max_retries=5
):

    region = (
        f"{chromosome}:"
        f"{position}-"
        f"{position}:1"
    )

    url = (
        f"{VEP_URL}/"
        f"{region}/"
        f"{alt}"
    )

    params = {
        "distance": DISTANCE,
        "canonical": 1,
        "mane": 1,
        "hgvs": 1,
        "check_existing": 1,
        "regulatory": 1
    }

    # --------------------------------------------------------
    # Retry loop
    # --------------------------------------------------------

    for attempt in range(1, max_retries + 1):

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                params=params,
                timeout=90
            )


            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            if response.status_code == 200:

                return response.json()


            # ------------------------------------------------
            # TEMPORARY SERVER ERRORS
            # ------------------------------------------------

            if response.status_code in [
                429,  # Too many requests
                500,  # Internal server error
                502,  # Bad gateway
                503,  # Service unavailable
                504   # Gateway timeout
            ]:

                wait_time = 2 ** attempt

                print(
                    f"\nTemporary VEP error "
                    f"{response.status_code} "
                    f"for {chromosome}:{position} {alt}"
                )

                print(
                    f"Retrying in "
                    f"{wait_time} seconds "
                    f"(attempt "
                    f"{attempt}/{max_retries})..."
                )

                time.sleep(
                    wait_time
                )

                continue


            # ------------------------------------------------
            # PERMANENT ERROR
            # ------------------------------------------------

            print(
                f"\nVEP failed permanently:"
            )

            print(
                f"{chromosome}:{position} {alt}"
            )

            print(
                "HTTP status:",
                response.status_code
            )

            print(
                response.text[:500]
            )

            return None


        except requests.exceptions.Timeout:

            wait_time = 2 ** attempt

            print(
                f"\nTimeout for "
                f"{chromosome}:{position} {alt}"
            )

            print(
                f"Retrying in "
                f"{wait_time} seconds..."
            )

            time.sleep(
                wait_time
            )


        except requests.exceptions.ConnectionError:

            wait_time = 2 ** attempt

            print(
                f"\nConnection error for "
                f"{chromosome}:{position} {alt}"
            )

            print(
                f"Retrying in "
                f"{wait_time} seconds..."
            )

            time.sleep(
                wait_time
            )


        except Exception as e:

            print(
                f"\nUnexpected error for "
                f"{chromosome}:{position}:"
            )

            print(e)

            return None


    # --------------------------------------------------------
    # ALL RETRIES FAILED
    # --------------------------------------------------------

    print(
        f"\nFAILED after "
        f"{max_retries} attempts:"
    )

    print(
        f"{chromosome}:{position} {alt}"
    )

    return None
# ============================================================
# PROCESS EACH SNP
# ============================================================

for i in tqdm(
    range(len(top39)),
    desc="Finding nearest genes"
):

    rsid = top39.loc[
        i,
        "RSID"
    ]

    chromosome = str(
        top39.loc[
            i,
            "CHR"
        ]
    )

    position = int(
        top39.loc[
            i,
            "POS"
        ]
    )

    alt = str(
        top39.loc[
            i,
            "ALT"
        ]
    )


    # ========================================================
    # QUERY VEP
    # ========================================================

    results = query_vep(
        chromosome,
        position,
        alt
    )


    if not results:
        continue


    variant = results[0]


    # ========================================================
    # FIND NEARBY GENES
    # ========================================================

    nearby = []


    for tc in variant.get(
        "transcript_consequences",
        []
    ):

        gene = tc.get(
            "gene_symbol"
        )

        gene_id = tc.get(
            "gene_id"
        )

        distance = tc.get(
            "distance"
        )

        consequences = tc.get(
            "consequence_terms",
            []
        )


        # Only retain records where VEP
        # provides a gene AND distance

        if (
            gene
            and distance is not None
        ):

            try:

                distance = abs(
                    int(distance)
                )

            except (
                ValueError,
                TypeError
            ):

                continue


            nearby.append(
                {
                    "gene": gene,
                    "gene_id": gene_id,
                    "distance": distance,
                    "consequence":
                        ";".join(
                            consequences
                        )
                }
            )


    # ========================================================
    # REMOVE DUPLICATES
    # ========================================================

    unique = {}

    for x in nearby:

        key = (
            x["gene"],
            x["gene_id"],
            x["distance"]
        )

        unique[key] = x


    nearby = list(
        unique.values()
    )


    # ========================================================
    # SORT BY DISTANCE
    # ========================================================

    nearby.sort(
        key=lambda x: x["distance"]
    )


    # ========================================================
    # SAVE RESULTS
    # ========================================================

    if nearby:

        # ----------------------------------------------------
        # All nearby genes
        # ----------------------------------------------------

        top39.loc[
            i,
            "Nearby_genes"
        ] = ";".join(
            unique_join(
                [
                    x["gene"]
                    for x in nearby
                ]
            ).split(";")
        )


        top39.loc[
            i,
            "Nearby_gene_distances_bp"
        ] = ";".join(
            [
                str(x["distance"])
                for x in nearby
            ]
        )


        # ----------------------------------------------------
        # Closest gene
        # ----------------------------------------------------

        closest = nearby[0]


        top39.loc[
            i,
            "Nearest_gene"
        ] = closest["gene"]


        if closest["gene_id"]:

            top39.loc[
                i,
                "Nearest_gene_ID"
            ] = closest["gene_id"]


        top39.loc[
            i,
            "Nearest_gene_distance_bp"
        ] = int(
            closest["distance"]
        )


        top39.loc[
            i,
            "Nearest_gene_consequence"
        ] = closest[
            "consequence"
        ]


    # ========================================================
    # API DELAY
    # ========================================================

    time.sleep(0.15)


# ============================================================
# SAVE
# ============================================================

top39.to_csv(
    OUTPUT,
    sep="\t",
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 70)
print("FINISHED")
print("=" * 70)

print(
    f"Output file: {OUTPUT}"
)

print(
    f"Number of SNPs: {len(top39)}"
)


has_gene = (
    top39["Nearest_gene"]
    .fillna("")
    .astype(str)
    .str.len()
    .gt(0)
)

print(
    f"SNPs with nearest gene: "
    f"{has_gene.sum()}/{len(top39)}"
)

print(
    f"SNPs without nearby gene "
    f"within {DISTANCE:,} bp: "
    f"{(~has_gene).sum()}/{len(top39)}"
)


# ============================================================
# SHOW RESULTS
# ============================================================

print("\nResults:")

print(
    top39[
        [
            "RSID",
            "CHR",
            "POS",
            "REF",
            "ALT",
            "Nearest_gene",
            "Nearest_gene_ID",
            "Nearest_gene_distance_bp",
            "Nearest_gene_consequence"
        ]
    ].to_string(
        index=False
    )
)

In [12]:
# merging all 3 into one final table:

import pandas as pd
import numpy as np

# ============================================================
# FILES
# ============================================================

TOP39 = "top39.txt"

NEAREST = "top39_nearest_gene.tsv"

TOPMED_GWAS = "top39_VEP_annotation_TOPMed_GWASCatalog.tsv"

VEP_MAF = "top39_vep_annotation_with_mafs.tsv"

OUTPUT = "top39_FINAL_annotation.tsv"


# ============================================================
# 1. LOAD ORIGINAL 39 SNP LIST
# ============================================================

top39 = pd.read_csv(
    TOP39,
    sep=r"\s+",
    header=None,
    dtype=str
)

# First 5 columns of top39.txt
top39 = top39.iloc[:, :5]

top39.columns = [
    "CHR",
    "POS",
    "RSID",
    "REF",
    "ALT"
]

# Remove accidental duplicate RSIDs
top39 = top39.drop_duplicates(
    subset="RSID",
    keep="first"
).reset_index(drop=True)

print(
    f"Original SNP list: {len(top39)} unique SNPs"
)


# ============================================================
# 2. LOAD ANNOTATION FILES
# ============================================================

nearest = pd.read_csv(
    NEAREST,
    sep="\t",
    dtype=str
)

topmed_gwas = pd.read_csv(
    TOPMED_GWAS,
    sep="\t",
    dtype=str
)

vep_maf = pd.read_csv(
    VEP_MAF,
    sep="\t",
    dtype=str
)


print("\nLoaded files:")
print(
    f"Nearest gene: {len(nearest)} rows"
)

print(
    f"TOPMed/GWAS Catalog: {len(topmed_gwas)} rows"
)

print(
    f"VEP + MAF: {len(vep_maf)} rows"
)


# ============================================================
# 3. CHECK RSID COLUMN
# ============================================================

for name, df in [
    ("nearest", nearest),
    ("topmed_gwas", topmed_gwas),
    ("vep_maf", vep_maf)
]:

    if "RSID" not in df.columns:

        raise ValueError(
            f"{name} does not contain an RSID column.\n"
            f"Columns found:\n{list(df.columns)}"
        )


# ============================================================
# 4. REMOVE DUPLICATE RSIDS FROM ANNOTATION TABLES
# ============================================================
#
# This is important because your VEP-derived files can contain
# multiple rows for one SNP.
#
# We want ONE row per RSID in the final table.
#
# ============================================================

def collapse_duplicate_rsids(df, name):

    n_before = len(df)

    duplicate_count = (
        df["RSID"]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:

        print(
            f"\n{name}: found "
            f"{duplicate_count} duplicate RSID rows."
        )

        print(
            df.loc[
                df["RSID"].duplicated(
                    keep=False
                ),
                "RSID"
            ]
            .drop_duplicates()
            .tolist()
        )

        # Keep first annotation record
        df = df.drop_duplicates(
            subset="RSID",
            keep="first"
        ).copy()

        print(
            f"{name}: "
            f"{n_before} -> {len(df)} rows"
        )

    return df


nearest = collapse_duplicate_rsids(
    nearest,
    "Nearest gene"
)

topmed_gwas = collapse_duplicate_rsids(
    topmed_gwas,
    "TOPMed/GWAS Catalog"
)

vep_maf = collapse_duplicate_rsids(
    vep_maf,
    "VEP + MAF"
)


# ============================================================
# 5. REMOVE REDUNDANT COLUMNS BEFORE MERGING
# ============================================================
#
# We keep the original top39.txt as the authoritative source
# for:
#
# CHR
# POS
# RSID
# REF
# ALT
#
# Therefore, if annotation files contain copies of these,
# don't create CHR_x, CHR_y, etc.
#
# ============================================================

MASTER_COLUMNS = [
    "CHR",
    "POS",
    "RSID",
    "REF",
    "ALT"
]


def remove_master_columns(
    df,
    name
):

    columns_to_remove = [
        c for c in MASTER_COLUMNS
        if c in df.columns
        and c != "RSID"
    ]

    if columns_to_remove:

        print(
            f"\nRemoving redundant columns "
            f"from {name}:"
        )

        print(
            columns_to_remove
        )

        df = df.drop(
            columns=columns_to_remove
        )

    return df


nearest = remove_master_columns(
    nearest,
    "Nearest gene"
)

topmed_gwas = remove_master_columns(
    topmed_gwas,
    "TOPMed/GWAS Catalog"
)

vep_maf = remove_master_columns(
    vep_maf,
    "VEP + MAF"
)


# ============================================================
# 6. IDENTIFY OVERLAPPING NON-RSID COLUMNS
# ============================================================

def show_overlap(
    df1,
    df2,
    name1,
    name2
):

    overlap = (
        set(df1.columns)
        & set(df2.columns)
    )

    overlap.discard("RSID")

    if overlap:

        print(
            f"\nOverlapping columns "
            f"between {name1} and {name2}:"
        )

        print(
            sorted(overlap)
        )


show_overlap(
    nearest,
    topmed_gwas,
    "Nearest",
    "TOPMed/GWAS"
)

show_overlap(
    nearest,
    vep_maf,
    "Nearest",
    "VEP/MAF"
)

show_overlap(
    topmed_gwas,
    vep_maf,
    "TOPMed/GWAS",
    "VEP/MAF"
)


# ============================================================
# 7. MERGE
# ============================================================
#
# Start with the authoritative 39-SNP table.
#
# LEFT JOIN means every SNP from top39.txt is retained.
#
# ============================================================

result = top39.copy()


# ------------------------------------------------------------
# Nearest gene
# ------------------------------------------------------------

result = result.merge(
    nearest,
    on="RSID",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# TOPMed + GWAS Catalog
# ------------------------------------------------------------

result = result.merge(
    topmed_gwas,
    on="RSID",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# VEP + MAF
# ------------------------------------------------------------

result = result.merge(
    vep_maf,
    on="RSID",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 8. REMOVE IDENTICAL DUPLICATE COLUMNS
# ============================================================
#
# If two annotation files contain columns with the same
# information but different names, this section catches
# columns whose entire contents are identical.
#
# ============================================================

columns = result.columns.tolist()

columns_to_drop = []

for i in range(len(columns)):

    col1 = columns[i]

    if col1 in columns_to_drop:
        continue

    for j in range(i + 1, len(columns)):

        col2 = columns[j]

        if col2 in columns_to_drop:
            continue

        try:

            identical = result[col1].equals(
                result[col2]
            )

        except Exception:

            identical = False


        if identical:

            print(
                f"\nRemoving redundant identical "
                f"column: {col2}"
            )

            print(
                f"  identical to: {col1}"
            )

            columns_to_drop.append(
                col2
            )


if columns_to_drop:

    result = result.drop(
        columns=columns_to_drop
    )


# ============================================================
# 9. REMOVE PANDAS _x / _y COLUMNS IF THEY EXIST
# ============================================================
#
# Normally the previous steps should prevent these, but this
# protects against unexpected overlaps.
#
# ============================================================

xy_columns = [
    c for c in result.columns
    if c.endswith("_x")
    or c.endswith("_y")
]

if xy_columns:

    print(
        "\nWARNING: _x/_y columns detected:"
    )

    print(
        xy_columns
    )


# ============================================================
# 10. FINAL COLUMN ORDER
# ============================================================

first_columns = [
    "CHR",
    "POS",
    "RSID",
    "REF",
    "ALT"
]

remaining_columns = [
    c for c in result.columns
    if c not in first_columns
]

result = result[
    first_columns
    + remaining_columns
]


# ============================================================
# 11. CHECK FINAL DATASET
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Rows:    {len(result)}"
)

print(
    f"Columns: {len(result.columns)}"
)


# Check that every RSID is unique
if result["RSID"].duplicated().any():

    print(
        "\nWARNING: duplicate RSIDs remain:"
    )

    print(
        result.loc[
            result["RSID"].duplicated(
                keep=False
            ),
            "RSID"
        ].tolist()
    )

else:

    print(
        "RSIDs:   all unique"
    )


# ============================================================
# 12. CHECK WHICH SNPs ARE MISSING ANNOTATIONS
# ============================================================

print("\nAnnotation coverage:")

for column in [
    "Nearest_gene",
    "Nearest_gene_ID",
    "TOPMed_AF",
    "GWAS_Catalog_Traits"
]:

    if column in result.columns:

        nonempty = (
            result[column]
            .notna()
            .sum()
        )

        print(
            f"{column}: "
            f"{nonempty}/{len(result)}"
        )


# ============================================================
# 13. SAVE
# ============================================================

result.to_csv(
    OUTPUT,
    sep="\t",
    index=False
)

print(
    f"\nSaved final annotation table:"
)

print(
    OUTPUT
)


# ============================================================
# 14. SHOW FINAL TABLE
# ============================================================

print("\nFirst rows:")

print(
    result.head(10).to_string(
        index=False
    )
)

Original SNP list: 39 unique SNPs

Loaded files:
Nearest gene: 39 rows
TOPMed/GWAS Catalog: 46 rows
VEP + MAF: 46 rows

TOPMed/GWAS Catalog: found 7 duplicate RSID rows.
['rs3795179', 'rs9263600']
TOPMed/GWAS Catalog: 46 -> 39 rows

VEP + MAF: found 7 duplicate RSID rows.
['rs3795179', 'rs9263600']
VEP + MAF: 46 -> 39 rows

Removing redundant columns from Nearest gene:
['CHR', 'POS', 'REF', 'ALT']

Overlapping columns between TOPMed/GWAS and VEP/MAF:
['AF', 'AFR', 'AMR', 'AlphaMissense', 'Amino_acids', 'Biotypes', 'CADD', 'Canonical_transcripts', 'Codons', 'Colocated_variants', 'Consequence_terms', 'EAS', 'EUR', 'Ensembl_gene_IDs', 'GWAS_A1', 'GWAS_ALT', 'GWAS_CHR', 'GWAS_POS', 'GWAS_REF', 'Gene_symbols', 'Impacts', 'Location_match', 'MANE_transcripts', 'Most_severe_consequence', 'PolyPhen', 'Protein_changes', 'SAS', 'SIFT', 'Transcript_IDs', 'VEP_Allele_string', 'VEP_Assembly', 'VEP_CHR', 'VEP_POS', 'gnomADg_AF', 'gnomADg_AFR', 'gnomADg_AF_Matches_GWAS_ALT', 'gnomADg_AMI', 'gnomADg_AM